# Prepare to share the predictions as a custom managed data product using the SAP BDC Connect SDK

In this notebook we want to publish the cashflow prediction results to the data catalog of SAP Business Data Cloud. For that we need to build a custom data product by using the python library sap-bdc-connect-sdk. We will also utilize the Delta Share protocol, which allows us to share the data product without the need of copying the result table. The result table remains persisted in SAP Databricks, and will be remotely accessible from the data catalog.

The following steps need to be applied in this exercise:

- Install and load packages
- Create a client for sap-bdc-connect-sdk and Create Delta Share
- Create Open Resource Discovery (ORD) object
- Create Core Schema Notation (CSN) object
- Publish data product


## Install and load Packages
To be able to share the enhanced data products back to SAP Business Data Cloud, we need the SAP SDK (https://pypi.org/project/sap-bdc-connect-sdk/)

In [0]:
%pip install typing-extensions=4.12.0
%pip install pydantic=2.9.2
%pip install sap-bdc-connect-sdk

## Create a client for sap-bdc-connect-sdk, create Delta Share and add recipient to Delta share

# &#x270D;
# USE YOUR OWN SCHEMA
In order to isolate the created data assets, we create a catalog within Databricks and a respective schema within the catalog. Please replace the values `<CATALOG_NAME>` and `<SCHEMA_NAME>` with the specific values that match our use case and group. You can find the correct names by checking the **Unity Catalog** and look for the specific catalog and schema names: here replace _<USERID> with your appropriate user id given to you. i.e. Your schema name will be cnr_AC2XXXXXXX01, cnr_AC2XXXXXXX02 and so on. 

In [0]:

%sql
-- CREATE CATALOG IF NOT EXISTS ;
SET CATALOG cnr_catalog;
CREATE SCHEMA IF NOT EXISTS cnr_<USERID>;
USE SCHEMA cnr_<USERID>;
     

# &#x270D;
# CREATE YOUR OWN SHARE
Replace _<USERID> with your appropriate user id given to you. i.e. Your share name will be cnr_scorecard_AC2XXXXXXX01, cnr_scorecard_AC2XXXXXXX02 and so on. 

In [0]:
%sql
CREATE SHARE cnr_scorecard_<USERID>;
ALTER SHARE cnr_scorecard_<USERID> ADD TABLE dp_dim_carbon_score WITH HISTORY;
ALTER SHARE cnr_scorecard_<USERID> ADD TABLE dp_dim_recommendation WITH HISTORY;
ALTER SHARE cnr_scorecard_<USERID> ADD TABLE dp_dim_supplier WITH HISTORY;
ALTER SHARE cnr_scorecard_<USERID> ADD TABLE dp_dim_time WITH HISTORY;
ALTER SHARE cnr_scorecard_<USERID> ADD TABLE dp_fact_carbon_emissions WITH HISTORY;
ALTER SHARE cnr_scorecard_<USERID> ADD TABLE dp_fact_risk_assessment WITH HISTORY;
ALTER SHARE cnr_scorecard_<USERID> ADD TABLE dp_fact_supplier_spend WITH HISTORY;

     

# &#x270D;
# USE YOUR OWN SHARE
Replace _<USERID> with your appropriate user id given to you. i.e. Your share name will be cnr_scorecard_AC2XXXXXXX01, cnr_scorecard_AC2XXXXXXX02 and so on. 

In [0]:
%sql
GRANT SELECT ON SHARE cnr_scorecard_<USERID> TO RECIPIENT `sap-business-data-cloud`;

## Create or update share

A share is a mechanism for distributing and accessing data across different systems. Creating or updating a share involves including specific attributes, such as @openResourceDiscoveryV1, in the request body, aligning with the Open Resource Discovery protocol. This procedure ensures that the share is properly structured and described according to specified standards, facilitating effective data sharing and management.

# &#x270D;
# CREATE YOUR OWN DATA PRODUCT
- Replace _<USERID> with your appropriate user id given to you in the data product name. i.e. Your data product name will be DDP_Supplier_scorecard_AC2XXXXXXX01, DDP_Supplier_scorecard_AC2XXXXXXX02 and so on. 
- Replace _<USERID> with your appropriate user id given to you in the share name. i.e. Your share name will be cnr_scorecard_AC2XXXXXXX01, cnr_scorecard_AC2XXXXXXX02 and so on. 

In [0]:
from bdc_connect_sdk.auth import BdcConnectClient
from bdc_connect_sdk.auth import DatabricksClient

bdc_connect_client = BdcConnectClient(DatabricksClient(dbutils, "sap-business-data-cloud"))

share_name = "cnr_scorecard_<USERID>"


open_resource_discovery_information = {
    "@openResourceDiscoveryV1": {
        "title": "DDP_Supplier_scorecard_<USERID>",
        "shortDescription": "The table contains data related supplier risk assessment and supplier scale predictions",
        "description": "The risk assessment is done based on sustainibility data and credit risk data per supplier"
            }
        }

response = bdc_connect_client.create_or_update_share(
    share_name,
    open_resource_discovery_information
)
print(f"[REQUEST] Create or update share request was executed and returned {response}")

## Create or update share CSN
The CSN serves as a standardized format for configuring and describing shares within a network. To create or update the CSN for a share, it's advised to prepare the CSN content in a separate file and include this content in the request body. This approach ensures accuracy and compliance with the CSN interoperability specifications, facilitating consistent and effective share configuration across systems.

_Note: The CSN schema, can be efficiently generated using a dedicated script generate_csn_template available in csn_generator file. This script automates the creation of CSN content for Databricks environments based on a Databricks share._

# &#x270D;
Replace _<USERID> with your appropriate user id given to you in the share name. i.e. Your share name will be cnr_scorecard_AC2XXXXXXX01, cnr_scorecard_AC2XXXXXXX02 and so on. 

In [0]:
from bdc_connect_sdk.auth import BdcConnectClient
from bdc_connect_sdk.auth import DatabricksClient
from bdc_connect_sdk.utils import csn_generator
import re

# Monkey-patch to fix FK constraint parsing bug in csn_generator:
# The library's regex captures all backtick-enclosed identifiers including
# catalog/schema/table names from FOREIGN KEY REFERENCES clauses, causing KeyError.
_original_get_table_column_mapping = csn_generator.get_table_column_mapping

def _patched_get_table_column_mapping(full_table_name, spark):
    from pyspark.sql import SparkSession
    import pandas
    from typing import Any
    from bdc_connect_sdk.utils.csn_generator import add_backticks

    spark_csn_mapping = {
        "boolean": "cds.Boolean",
        "string": "cds.String",
        "varchar": "cds.String",
        "int": "cds.Integer",
        "double": "cds.Double",
        "bigint": "cds.Integer64",
        "decimal": "cds.Decimal",
        "date": "cds.Date",
        "timestamp": "cds.DateTime",
        "timestamp_ms": "cds.Timestamp"
    }

    pandas.set_option('display.max_colwidth', None)
    pandas.set_option('display.max_columns', None)
    pandas.set_option('display.max_rows', None)

    schema_sql = f"DESCRIBE TABLE EXTENDED {add_backticks(full_table_name)}"
    schema_description = spark.sql(schema_sql).toPandas()
    schema_list = schema_description.values.tolist()

    column_mapping: dict[str, Any] = {}
    is_column_section = True
    is_constraints_section = False
    for column_name, data_type, _ in schema_list:
        if not column_name and not data_type:
            is_column_section = False
            is_constraints_section = False
            continue

        if "Constraints" in column_name:
            is_constraints_section = True
            continue

        if is_column_section:
            if "decimal" in data_type:
                schema_key = data_type.split("(")[0]
                precision, scale = re.findall(r"\d+", data_type)
                base_dtype = spark_csn_mapping[schema_key]
                column_mapping[column_name] = {"type": base_dtype, "precision": int(precision), "scale": int(scale)}
            elif "string" in data_type:
                schema_key = "string"
                base_dtype = spark_csn_mapping[schema_key]
                column_mapping[column_name] = {"type": base_dtype, "length": 5000}
            elif "varchar" in data_type:
                schema_key = "varchar"
                length = re.findall(r"\d+", data_type)[0]
                base_dtype = spark_csn_mapping[schema_key]
                column_mapping[column_name] = {"type": base_dtype, "length": int(length)}
            elif "float" in data_type:
                raise TypeError("Unsupported data type(float): consider data type `Double`.")
            else:
                csn_dtype = spark_csn_mapping[data_type]
                column_mapping[column_name] = {"type": csn_dtype}

        if is_constraints_section:
            if "PRIMARY" in data_type or "FOREIGN" in data_type:
                # Fix: only extract columns from the part BEFORE REFERENCES
                local_part = data_type.split("REFERENCES")[0]
                columns = re.findall(r'`([^`]+)`', local_part)
                for col in columns:
                    if col in column_mapping:
                        column_mapping[col]["key"] = True

    return column_mapping

csn_generator.get_table_column_mapping = _patched_get_table_column_mapping

bdc_connect_client = BdcConnectClient(DatabricksClient(dbutils, "sap-business-data-cloud"))

share_name = "cnr_scorecard_<USERID>"

csn_schema = csn_generator.generate_csn_template(share_name)

response = bdc_connect_client.create_or_update_share_csn(
    share_name,
    csn_schema
)
print(f"[REQUEST] Create or update CSN request was executed and returned {response if response else 'OK'}")

### Publish a Data Product
A Data Product is an abstraction that represents a type of data or data set within a system, facilitating easier management and sharing across different platforms. It bundles resources or API endpoints to enable efficient data access and utilization by integrated systems. Publishing a Data Product allows these systems to access and consume the data, ensuring seamless communication and resource sharing.

# &#x270D;
Replace _<USERID> with your appropriate user id given to you in the share name. i.e. Your share name will be cnr_scorecard_AC2XXXXXXX01, cnr_scorecard_AC2XXXXXXX02 and so on. 

In [0]:
from bdc_connect_sdk.auth import BdcConnectClient
from bdc_connect_sdk.auth import DatabricksClient
from bdc_connect_sdk.utils import csn_generator

bdc_connect_client = BdcConnectClient(DatabricksClient(dbutils, "sap-business-data-cloud"))

share_name = "cnr_scorecard_<USERID>"

response = bdc_connect_client.publish_data_product(
    share_name
)
print(f"[REQUEST] Publish Data Product request was executed and returned {response if response else 'OK'}")